# CS5489 Course Project — AIGC Localization

**Group name:** [Canvas group name]  
**Members:** [Member names]

We predict a binary mask of edited pixels from each RGB image. This notebook documents the frozen Weighted BCE candidate. Run all cells to inspect the measured results. Enable the last cell's training toggle to reproduce the model with the official labelled archive.

## Data, model, and selection

The fixed split contains 4,500 training and 500 validation images. We resize full RGB images to 256 × 256 with bilinear interpolation and scale them to [0, 1]. Binary training masks use nearest-neighbor resizing. A randomly initialized U-Net uses two max-pool stages and two skip connections. The experiment uses course data and random initialization.

The training objective combines positive-weighted binary cross-entropy with per-image soft Dice loss. The positive weight is the square root of the ratio of background to foreground training pixels. We use Adam at a learning rate of 0.001, batch size 8, seed 5489 and 10 epochs. The checkpoint with the highest 256 × 256 validation micro-IoU at threshold 0.5 is retained. At inference, logits are resized to the original image shape before thresholding.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

config = json.loads(Path("config.json").read_text())
history = pd.read_csv("training_history.csv")
summary = json.loads(Path("training_summary.json").read_text())
assert history.loc[history.val_iou.idxmax(), "epoch"] == summary["best_epoch"]
print(f'Positive BCE weight: {config["bce_pos_weight"]:.6f}')
print(f'Best checkpoint: epoch {summary["best_epoch"]}, 256-grid IoU {summary["best_val_iou_256"]:.2%}')
display(history)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), layout="constrained")
for ax, column, label in zip(
    axes, ("train_weighted_loss", "val_loss", "val_iou"),
    ("Weighted training loss", "Unweighted validation loss", "Validation micro-IoU"),
):
    ax.plot(history.epoch, history[column], "o-")
    ax.set(xlabel="Epoch", ylabel=label)
    ax.grid(alpha=0.25)
plt.show()

## Original-resolution validation

Micro-IoU pools the pixel counts over all 500 images. Mean image IoU weights images equally. Both-empty prediction/target pairs would score 1 in our local protocol. All validation masks contain foreground.

In [ ]:
counts = pd.read_csv("validation_by_image.csv", dtype={"ID": str})
tp, fp, fn = counts[["TP", "FP", "FN"]].sum()
comparison = pd.read_csv("historical_comparison.csv")
assert len(counts) == counts.ID.nunique() == 500
assert ((counts.TP + counts.FN) > 0).all()
assert abs(tp / (tp + fp + fn) - comparison.loc[comparison.model.eq("Weighted"), "iou"].item()) < 1e-12
print(f"Uniform foreground micro-IoU: {(tp + fn) / counts.pixels.sum():.2%}")
small = counts.loc[counts.target_fraction.lt(.01)]
print(f"Weighted BCE IoU for edits below 1% ({len(small)} images): {small.TP.sum() / small[['TP', 'FP', 'FN']].to_numpy().sum():.2%}")
display(comparison[["model", "iou", "mean_image_iou", "dice", "precision", "recall", "FP", "FN"]])

Weighted BCE increased recall and false positives relative to the unweighted 256 × 256 model. It led the three archived checkpoints on micro-IoU and Dice, while the 384 × 384 candidate led on mean image IoU. Localization of small edits remains weak. Model selection uses the held-out training split. The public images serve to verify the inference pipeline.

## Reproduce training

Place the official ZIP beside this notebook or edit its path below. The training command creates a cache with the same preprocessing and writes a new checkpoint to reproduction/model.pth. The frozen model.pth serves test.ipynb.

In [ ]:
import subprocess
import sys

RETRAIN = False
ARCHIVE = Path("AIGC Detection Dataset 2026.zip")
if RETRAIN:
    subprocess.run([sys.executable, "-B", "train_candidate.py", "--archive", str(ARCHIVE)], check=True)